In [21]:
import os
import pandas as pd
from tqdm import tqdm
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

PINECONE_API_KEY = "pcsk_7TpTGL_UWECb2vZvBeB9mC9SbBuxLnQTNiJGj6jA3LpSg7YgQrrKbEVJ5ixoESDJ9GvFVM"
# PINECONE_API_KEY = "pcsk_5eB5sf_D4PKMqs9oGJVJ7vXqazVSYKiQMqZNFJqvYjFdtrfrBBHaPfrmFMSWnXTzkmFqeU"

INDEX_NAME = "test1"
CSV_PATH = "country.csv"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM = 384

In [14]:
model = SentenceTransformer(MODEL_NAME)

In [15]:
pc = Pinecone(api_key=PINECONE_API_KEY)

In [22]:
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric="cosine"
    )

In [23]:
index = pc.Index(INDEX_NAME)

In [24]:
df = pd.read_csv(CSV_PATH)

print(df.head())

             Name Code
0     Afghanistan   AF
1   Åland Islands   AX
2         Albania   AL
3         Algeria   DZ
4  American Samoa   AS


In [25]:
vectors = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    vector_id = str(row["Code"])
    text = row["Name"]

    embedding = model.encode(text).tolist()

    vectors.append({
        "id": vector_id,
        "values": embedding,
        "metadata": {
            "text": text
        }
    })


100%|██████████| 249/249 [00:05<00:00, 47.35it/s]


In [26]:
index.upsert(vectors=vectors)

UpsertResponse(upserted_count=249, _response_info={'raw_headers': {'date': 'Wed, 31 Dec 2025 21:38:47 GMT', 'content-type': 'application/json', 'content-length': '21', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-request-logical-size': '388783', 'x-pinecone-request-latency-ms': '3326', 'x-pinecone-request-id': '8928916504922760686', 'x-envoy-upstream-service-time': '304', 'x-pinecone-response-duration-ms': '3328', 'grpc-status': '0', 'server': 'envoy'}})

In [27]:
print(index.describe_index_stats())

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '185',
                                    'content-type': 'application/json',
                                    'date': 'Wed, 31 Dec 2025 21:39:09 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '3',
                                    'x-pinecone-request-id': '5721854612902036796',
                                    'x-pinecone-request-latency-ms': '3',
                                    'x-pinecone-response-duration-ms': '4'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 249}},
 'storageFullness': 0.0,
 'total_vector_count': 249,
 'vector_type': 'dense'}


In [28]:
query_text = "asian countries"
query_embedding = model.encode(query_text).tolist()

results = index.query(
    vector=query_embedding,
    top_k=3,
    include_metadata=True
)

for match in results["matches"]:
    print(match["score"], match["metadata"]["text"])


0.700525284 Japan
0.68437767 China
0.680588722 Thailand


In [29]:
import os
from tqdm import tqdm
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone


In [30]:
PINECONE_API_KEY = "pcsk_7TpTGL_UWECb2vZvBeB9mC9SbBuxLnQTNiJGj6jA3LpSg7YgQrrKbEVJ5ixoESDJ9GvFVM"
INDEX_NAME = "test1"   # same index
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

PDF_PATH = "277_response_understanding.pdf"


In [31]:
model = SentenceTransformer(MODEL_NAME)
EMBED_DIM = model.get_sentence_embedding_dimension()


In [32]:
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)


In [33]:
def extract_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    pages = []

    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            pages.append({
                "page": page_num + 1,
                "text": text
            })
    return pages


In [34]:
def chunk_text(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks


In [35]:
pdf_pages = extract_pdf_text(PDF_PATH)

vectors = []
doc_id = os.path.basename(PDF_PATH)

for page in pdf_pages:
    chunks = chunk_text(page["text"])

    for i, chunk in enumerate(chunks):
        embedding = model.encode(chunk).tolist()

        vector_id = f"{doc_id}_page{page['page']}_chunk{i}"

        vectors.append({
            "id": vector_id,
            "values": embedding,
            "metadata": {
                "source": "pdf",
                "document": doc_id,
                "page": page["page"],
                "chunk": i,
                "text": chunk
            }
        })


In [36]:
index.upsert(vectors=vectors)


UpsertResponse(upserted_count=16, _response_info={'raw_headers': {'date': 'Wed, 31 Dec 2025 21:52:13 GMT', 'content-type': 'application/json', 'content-length': '20', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '2', 'x-pinecone-request-logical-size': '32855', 'x-pinecone-request-latency-ms': '1970', 'x-pinecone-request-id': '1699150143206896139', 'x-envoy-upstream-service-time': '287', 'x-pinecone-response-duration-ms': '1972', 'grpc-status': '0', 'server': 'envoy'}})

In [38]:
query_text = "Why was CPT 99213 denied?"
query_embedding = model.encode(query_text).tolist()

results = index.query(
    vector=query_embedding,
    top_k=5,
    include_metadata=True
)

for match in results["matches"]:
    print("Score:", match["score"])
    # print("Source:", match["metadata"]["source"])
    print("Text:", match["metadata"]["text"])
    print("-" * 50)


Score: 0.46886152
Text: Confidential and Proprietary I 10 Interpreting the 277CA, Claim-level rejection • This is an example of a file that rejected a claim for invalid total charge. View the explanations on the next few pages for help in interpreting this report.
--------------------------------------------------
Score: 0.437035561
Text: Understanding the 277CA Claims Acknowledgement For X12N 837 electronic claim files only Updated: 10/15/2025
--------------------------------------------------
Score: 0.393551826
Text: Confidential and Proprietary I 4 When is the 277CA available? • The 277CA is created after your claim file generates the 999. If the 999 rejected, a 277CAwill not be generated. • See the 999 Training Module for more information on the 999 report. • Depending on the volume of files being received, it could take up to 25 minutes to receive this report after a file is submitted. • The 277CA is available for you to retrieve for 60 calendar days only. • This report is availab